In [3]:
from huggingface_hub import login
from dotenv import load_dotenv

login()

load_dotenv("../.env")

True

# Using `tool` decorator

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, tool

# Let's pretend we have a function that fetches the highest-rated catering services.
@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.

    Args:
        query: A search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    # Find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)

    return best_service


agent = CodeAgent(tools=[catering_service_tool], model=InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct"))

# Run the agent to find the best catering service
result = agent.run(
    "Can you give me the name of the highest-rated catering service in Gotham City?"
)

print(result)   # Output: Gotham Catering Co.

# Defining a Tool Class

In [ ]:
from smolagents import Tool, CodeAgent, InferenceClientModel

class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""

    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic Gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }

        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")

# Instantiate the tool
party_theme_tool = SuperheroPartyThemeTool()
agent = CodeAgent(tools=[party_theme_tool], model=InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct"))

# Run the agent to generate a party theme idea
result = agent.run(
    "What would be a good superhero party idea for a 'villain masquerade' theme?"
)

print(result)  # Output: "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains."

# Importing tool from the hub

In [ ]:
from smolagents import load_tool, CodeAgent, InferenceClientModel

image_generation_tool = load_tool(
    "m-ric/text-to-image",
    trust_remote_code=True
)

agent = CodeAgent(
    tools=[image_generation_tool],
    model=InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")
)

agent.run("Generate an image of a luxurious superhero-themed party at Wayne Manor with made-up superheros.")

# Importing a HuggingFace Space as a Tool

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, Tool

image_generation_tool = Tool.from_space(
    "black-forest-labs/FLUX.1-schnell",
    name="image_generator",
    description="Generate an image from a prompt"
)

model = InferenceClientModel("Qwen/Qwen2.5-Coder-32B-Instruct")

agent = CodeAgent(tools=[image_generation_tool], model=model)

agent.run(
    "Improve this prompt, then generate an image of it.",
    additional_args={'user_prompt': 'A grand superhero-themed party at Wayne Manor, with Alfred overseeing a luxurious gala'}
)

# Importing a LangChain Tool

In [ ]:
from ast import mod

from langchain_community.agent_toolkits.load_tools import load_tools
from smolagents import CodeAgent, InferenceClientModel, Tool

search_tool = Tool.from_langchain(load_tools(["serpapi"])[0])

agent = CodeAgent(tools=[search_tool], model=InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct"))

agent.run("Search for luxury entertainment ideas for a superhero-themed event, such as live performances and interactive experiences.")

#  Importing a tool collection from any MCP server

In [4]:
import os
from smolagents import ToolCollection, CodeAgent
from mcp import StdioServerParameters
from smolagents import InferenceClientModel


model = InferenceClientModel("Qwen/Qwen2.5-Coder-32B-Instruct")


server_parameters = StdioServerParameters(
    command="uvx",
    args=["--with", "mcp<2", "pubmedmcp@0.1.3"],
    env={"UV_PYTHON": "3.12", **os.environ},
)

with ToolCollection.from_mcp(server_parameters, trust_remote_code=True) as tool_collection:
    agent = CodeAgent(tools=[*tool_collection.tools], model=model, add_base_tools=True)
    agent.run("Please find a remedy for hangover.")

/home/nerfeddoge/miniconda3/envs/hf_agent_course/lib/python3.14/contextlib.py:141: FutureWarning: Parameter 'structured_output' was not specified. Currently it defaults to False, but in version 1.25, the default will change to True. To suppress this warning, explicitly set structured_output=True (new behavior) or structured_output=False (legacy behavior). See documentation at https://huggingface.co/docs/smolagents/tutorials/tools#structured-output-and-output-schema-support for more details.
  return next(self.gen)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Please find a remedy for hangover.                                                                              │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  hangover_remedies = web_search(query="hangover remedies")                                                        
  print(hangover_remedies)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Hangover remedies](https://en.wikipedia.org/wiki/Hangover_remedies)
Various foods, dishes, and medicines have been described as having a theoretical potential for easing or 
alleviating symptoms of a hangover.While recommendations and folk cures for foods and drinks to relieve hangover 
symptoms abound, hangover foods have not been scientifically proven to function as a remedy or cure for the 
hangover.In a review assessing eight randomised controlled trials of propranolol, tropisetron, tolfenamic acid, 
fructose/glucose, a yeast preparation and supplements containing Borago officinalis, Cynara scolymus and Opuntia 
ficus-indica, researchers concluded that "no compelling evidence exists to suggest that any conventional or 
complementary intervention is effective for preventing or treating alcohol hangover."

[Hangover remedies](https://grokipedia.com/page/Hangover_remedies)
Hangover remedies encompass a wide array of strategies, substances, and commercial products intended to alleviate 
or prevent the unpleasant physiological and psychological symptoms that occur after excessive alcohol consumption, 
a condition known as…

[7 ways to cure your hangover - Harvard 
Health](https://www.health.harvard.edu/healthy-aging-and-longevity/7-ways-to-cure-your-hangover)
May 22, 2023 · Suffering from a hangover? Here are some at-home remedies you can try to feel better in no time.

[16 Science-Backed Ways to Get Rid of a Hangover Faster - wikiHowHangover Cures: Effective Home Remedies (Supported
by Science)11 Fast-Acting Natural Hangover Remedies That Actually Work12 Home Remedies for Hangover & Self-Care 
Tips - eMediHealth18 of the best hangover cures that actually workHow Do You Get Over a Hangover in 10 Minutes? 11 
Tips](https://www.wikihow.com/Get-Rid-of-a-Hangover)
Everyone has had a night where they drank a little too much and woke up the next morning with a headache and a 
whole lot of regrets. For some of us, it’s nausea and a churning stomach. For others, it’s a pounding headache and 
an aversion to loud voices and sunlight. Whatever it is that’s plaguing you, there are things you can do to feel 
better. Here are 16 ways to feel better when you’ve got a nasty hangover. See full list on wikihow.com The most 
obvious solution, yet one of the best, is to pour yourself a tall glass of water and get to drinking. One of the 
reasons you feel so headachy, nauseous, and light-headed is because alcohol causes your body to lose lots of water,
causing dehydration. Drinking water throughout the day can help you to rehydrate and recover the water you’ve lost.
This can dramatically improve your headache. While we’ve probably caught you the day after, anytime you can do this
before you fall asleep on a night of drinking, you can lessen your hangover symptoms the next day. See full list on
wikihow.com Isotonic sports drinks, like Gatorade or Powerade, are another great way to replace your lost fluids. 
These drinks contain electrolytes, which are important for you to replace when you’re dehydrated. Although it’s 
tempting to try to wake yourself up with some caffeine, it's better to avoid the caffeinated sports drinks, as they
can further dehydrate you and even make the hangover worse. Vitamin packets — or oral rehydration packets — are 
another great thing to toss into some water for a pick me up. See full list on wikihow.com This might sound a 
little out there, but one study found that people who drank fruit juice had less severe thirst and headache 
symptoms. This could be because fruit juices are generally high in fructose, which is a sugar that affects your 
energy and your liver function. Your liver is in charge of processing the alcohol you drank, so juice might improve
how quickly your body metabolizes alcohol. See full list on wikihow.com Ginger tea is an excellent way to combat 
nausea and vomiting. It’s often used by pregnant women when they’re dealing with morning sickness. One option is to
boil 

[Step 1: Duration 10.42 seconds| Input tokens: 2,471 | Output tokens: 66]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Water")                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Water

[Step 2: Duration 3.05 seconds| Input tokens: 7,050 | Output tokens: 129]